# Midterm Document — Screenshot & Copy-Paste Guide

**How to use this notebook:**

1. **Quote (bar-icon) blocks** = text to copy-paste directly into the docx. Click the rendered quote text and copy — the `>` blockquote icon renders as a vertical bar.
2. **Code cells** = the exact code to screenshot in `oop.ipynb` (guaranteed same content). Run `python execute_notebook.py` first so outputs are visible, then screenshot each listed cell with its output, cropped to the cell.
3. Each code cell is followed by its figure caption to type under the screenshot.

---
# 1. Project Overview

> This project performs end-to-end dataset inspection and preprocessing of the Space Missions dataset (4,630 launches, 9 original features) to prepare a machine-learning-ready classification matrix for predicting launch success. The work is organized as Parts A–H in a single Jupyter notebook: dataset inspection (A), problem definition (B), data cleaning (C), feature transformation (D), feature engineering (E), data-leakage auditing (F), feature/target separation (G), and train/test preparation with leak-safe preprocessing (H). The final output is a saved set of model-ready NumPy arrays (X_train.npy, X_test.npy, y_train.npy, y_test.npy and feature_names.txt).
> >
> > Limitation: Price was 72.68% missing and was dropped rather than imputed, and preprocessing deliberately excludes post-launch information to avoid target leakage.


---
# 2. Data and Project Inputs

> Input: space_missions.csv — 4,630 rows × 9 columns. Columns: Company, Location, Date, Time, Rocket, Mission, RocketStatus, MissionStatus, Price. Span: 1957-10-04 to 2022-07-29. Missing values: Price 72.68%, Time 2.72%; 1 exact duplicate row. The raw file is never written to — all cleaning happens on a working copy (df).


## Screenshot — Raw dataset inspection (Part A)

In [ ]:
import numpy as np
import pandas as pd

# Load dataset while keeping raw reference intact
df_raw = pd.read_csv("space_missions.csv", encoding="latin1")
df = df_raw.copy()

# Part A: Dataset Inspection Outputs
print(f"Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns")
print("\n--- Missing Values Count ---")
print(df.isna().sum()[df.isna().sum() > 0])
print(f"\nExact Duplicate Rows: {df.duplicated().sum()}")

# Target Variable Distribution
print("\n--- Target Variable Distribution (MissionStatus) ---")
status_dist = pd.DataFrame(
    {
        "Count": df["MissionStatus"].value_counts(),
        "Percentage": (df["MissionStatus"].value_counts(normalize=True) * 100)
        .round(2)
        .astype(str)
        + "%",
    }
)
print(status_dist)

> **Caption:** Raw dataset inspection (Part A)


## Screenshot — Cleaned dataset preview

In [ ]:
# Preview the cleaned dataset.
print("Cleaned dataset - first 5 rows:")
display(df.head())

print("\nFinal column types:")
print(df.dtypes.to_string())

> **Caption:** Cleaned dataset preview


## Screenshot — Cleaning summary and integrity checks

In [ ]:
# =====================================================================
# C.4 - Cleaning summary and integrity check
# =====================================================================

summary = pd.DataFrame([
    {"Step": "Start (raw)", "Rows": len(df_raw), "Columns": df_raw.shape[1]},
    {"Step": "After RULE 1 (exact duplicates)", "Rows": 4629, "Columns": 9},
    {"Step": "After RULE 2 (business duplicate)", "Rows": len(df), "Columns": 9},
    {"Step": "After RULE 5 (Price dropped)", "Rows": len(df), "Columns": 8},
    {"Step": "After RULE 6 (Location dropped)", "Rows": len(df), "Columns": len(df.columns)},
]).set_index("Step")

summary["Rows_removed"] = summary["Rows"].shift(1) - summary["Rows"]
display(summary)

print(f"Total rows removed : {len(df_raw) - len(df)} (1 exact dup + 1 business dup)")
print(f"Columns            : {df_raw.shape[1]} raw -> {len(df.columns)} cleaned")
print("                     (+ Country extracted, - Price, - Location)")
print(f"Final shape        : {df.shape}")

# Integrity assertion: nothing else went missing.
assert len(df) == len(df_raw) - 2, "Unexpected row count change!"
print("\nPASS: exactly 2 rows removed, both documented duplicates.")

> **Caption:** Cleaning summary and integrity checks


---
# 3. Project Methodology and Development Process

## Phase 1 — Preparation and Foundation (Parts A–C: inspection & cleaning)

> Phase 1 inspected the raw CSV and cleaned it using eight documented rules: Date parsed to real datetime (0 unparseable, range 1957–2022), 1 exact duplicate removed (4,630 → 4,629), 1 business duplicate removed requiring a missing Time as the duplicate signature (→ 4,628), Price repaired from text-to-float with 0 values lost, Price column dropped (72.68% missing, above the 50–70% drop threshold), Country extracted from Location (22 distinct countries), and Time deliberately left unimputed and excluded from features.


## Screenshot — Duplicate removal (Rule 1)

In [ ]:
# =====================================================================
# RULE 1 - Remove exact duplicate rows
# =====================================================================
# An exact duplicate is a row where EVERY column matches another row. It
# carries no extra information and would double-count one launch.
before = len(df)
n_exact = int(df.duplicated(keep="first").sum())
df = df.drop_duplicates(keep="first").reset_index(drop=True)

print(f"Exact duplicates found   : {n_exact}")
print(f"Rows before -> after     : {before} -> {len(df)}")

# Show what was removed, for the audit trail.
exact_mask = df_raw.duplicated(keep=False)
print("\nThe duplicated record was:")
print(df_raw[exact_mask][["Company", "Date", "Rocket", "Mission", "MissionStatus"]].to_string())

> **Caption:** Duplicate removal (Rule 1)


## Screenshot — Price repair (Rule 4)

In [ ]:
# =====================================================================
# RULE 4 - Repair Price: text with thousands separators -> float
# =====================================================================
# Values look like '1,160.00' and '5,000.00'. We strip the comma and cast.
# CRITICAL: we verify that the parse does not silently destroy values
# (guide §9A warns about unparseable text quietly becoming 0).

raw_nonnull = int(df["Price"].notna().sum())

df["Price"] = pd.to_numeric(
    df["Price"].astype(str).str.replace(",", "", regex=False).str.strip(),
    errors="coerce",
)

parsed_nonnull = int(df["Price"].notna().sum())

print(f"Non-null Price before parse : {raw_nonnull}")
print(f"Non-null Price after parse  : {parsed_nonnull}")
print(f"Values lost                 : {raw_nonnull - parsed_nonnull}")
print(f"New dtype                   : {df['Price'].dtype}")

assert raw_nonnull == parsed_nonnull, "Price parsing destroyed information!"
print("\nPASS: every non-null Price value survived as a real number.")

> **Caption:** Price repair (Rule 4)


## Screenshot — Column drop decision (Rule 5)

In [ ]:
# =====================================================================
# RULE 5 - Drop Price (72.68% missing)
# =====================================================================
# Above the guide's 50-70% threshold. Imputing three-quarters of a column
# would fabricate precision rather than recover information.

missing_pct = df["Price"].isna().mean() * 100
print(f"Price missing: {missing_pct:.2f}%")

df = df.drop(columns=["Price"])
print(f"Price dropped. Columns now: {df.shape[1]}")
print(f"Remaining columns: {list(df.columns)}")

> **Caption:** Column drop decision (Rule 5)


## Screenshot — Missing-value threshold analysis

In [ ]:
missing_pct = (df_raw.isna().mean() * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.barh(missing_pct.index, missing_pct.values,
               color=["#C44E52" if v > 50 else "#4C72B0" for v in missing_pct.values],
               edgecolor="white", linewidth=1.2)

for bar, (col_name, v) in zip(bars, missing_pct.items()):
    n_missing = int(df_raw[col_name].isna().sum())
    ax.text(v + 1.0, bar.get_y() + bar.get_height() / 2,
            f"{v:.2f}%  ({n_missing:,} rows)", va="center", fontsize=10)

ax.axvline(50, ls="--", color="gray", linewidth=1.4)
ax.text(50.8, -0.45, "50% threshold\n(guide §8A)", fontsize=9, color="gray")
ax.set_xlabel("Percentage of values missing (%)", fontsize=11)
ax.set_title("Missingness before cleaning: Price was the decisive case",
             fontsize=12, fontweight="bold")
ax.set_xlim(0, 88)
plt.tight_layout()
plt.show()

print(f"Price was {missing_pct['Price']:.2f}% missing - above the guide's 50-70% drop")
print("threshold, so imputation would have fabricated more than it recovered.")
print(f"Time was only {missing_pct['Time']:.2f}% missing - low enough to keep, but with")
print("no causal bearing on reliability, so it was excluded rather than imputed.")

> **Caption:** Missing-value threshold analysis


## Screenshot — Visual: what cleaning removed

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: row counts through the cleaning steps
steps = ["Raw", "After exact\ndedup", "After business\ndedup", "Final"]
counts = [4630, 4629, 4628, len(df)]
colors = ["#8172B2", "#4C72B0", "#4C72B0", "#55A868"]
bars = axes[0].bar(steps, counts, color=colors, edgecolor="white", linewidth=1.2)
for bar, v in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width() / 2, v + 0.6,
                 f"{v:,}", ha="center", fontsize=10, fontweight="bold")
axes[0].set_ylim(4620, 4635)
axes[0].set_ylabel("Number of rows", fontsize=11)
axes[0].set_title("Cleaning removed only 2 rows", fontsize=12, fontweight="bold")
axes[0].tick_params(axis="x", labelsize=9)

# Right: what we deliberately KEPT
kept = pd.Series({
    "Legitimate\nco-launches\n(KEPT)": 8,
    "Business\nduplicate\n(removed)": 1,
    "Exact\nduplicate\n(removed)": 1,
})
bars2 = axes[1].bar(kept.index, kept.values,
                    color=["#55A868", "#C44E52", "#C44E52"],
                    edgecolor="white", linewidth=1.2)
for bar, v in zip(bars2, kept.values):
    axes[1].text(bar.get_x() + bar.get_width() / 2, v + 0.25, str(v),
                 ha="center", fontsize=11, fontweight="bold")
axes[1].set_ylabel("Number of rows", fontsize=11)
axes[1].set_title("Rows sharing Company+Rocket+Date", fontsize=12, fontweight="bold")
axes[1].set_ylim(0, 10)
axes[1].tick_params(axis="x", labelsize=9)

plt.tight_layout()
plt.show()

print("A naive 'same company + rocket + date = duplicate' rule would have")
print("deleted all 10 rows. Requiring a MISSING TIME as the duplicate signature")
print("removed only the 1 true re-entry and preserved 8 real launches.")

> **Caption:** Visual: what cleaning removed


## Phase 2 — Development and Analysis (Parts D–F)

> Phase 2 created the feature space. Part D derived Launch_Year and cyclical month encoding (Month_Sin / Month_Cos, verified so December–January measure as adjacent), demonstrated why scaling matters for distance-based models but not for trees, and showed one-hot encoding of Rocket (370 distinct values → 370 columns) is infeasible, motivating frequency encoding.
> >
> > Part E implemented OOP feature builders — an abstract FeatureBuilder base class (ABC, cannot instantiate without implementing the interface) and a concrete OperationalExperienceBuilder producing Company_Prior_Launches and Company_Prior_Failure_Rate. A leaky version of the experience feature was found to inflate F1 by +49.6%.
> >
> > Part F audited leakage with a LeakageGuard class and proved three latent leaks in the raw columns: RocketStatus is temporally encoded (all 1,837 pre-1980 launches are Retired), Mission names encode outcome (test missions), and Price is recorded retroactively (33/774 priced in the 1960s vs 528/676 in the 2010s).


## Screenshot — Date/time feature extraction (D.2)

In [ ]:
# =====================================================================
# D.2 - Date/time extraction
# =====================================================================
# Two kinds of feature are derived from Date:
#
#   (a) Launch_Year  - ordinal, captures the technological era
#   (b) Month_Sin / Month_Cos - CYCLICAL encoding of month
#
# Why cyclical for month? Left as a raw integer, December (12) and January (1)
# appear MAXIMALLY distant - 11 units apart - when they are actually adjacent
# months. That is backwards. Projecting the month onto a unit circle fixes it.
#
# Both sine AND cosine are required: a single sine value is ambiguous, because
# two different months can share the same sine. Together they give each month
# a unique position.

df["Launch_Year"] = df["Date"].dt.year
df["Month_Sin"] = np.sin(2 * np.pi * df["Date"].dt.month / 12)
df["Month_Cos"] = np.cos(2 * np.pi * df["Date"].dt.month / 12)

print("New columns created: Launch_Year, Month_Sin, Month_Cos\n")
display(df[["Date", "Launch_Year", "Month_Sin", "Month_Cos"]].head(6))

print(f"Launch_Year range: {df['Launch_Year'].min()} - {df['Launch_Year'].max()}")

> **Caption:** Date/time feature extraction (D.2)


## Screenshot — Visual: sin/cos month encoding vs naive integer

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# --- Left: the WRONG representation (raw integers on a line) ---
months = np.arange(1, 13)
labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
          "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
axes[0].scatter(months, np.zeros_like(months), s=260, color="#4C72B0", zorder=3)
for m, lab in zip(months, labels):
    axes[0].annotate(lab, (m, 0), textcoords="offset points", xytext=(0, 14),
                     ha="center", fontsize=10)
axes[0].annotate("", xy=(12, 0), xytext=(1, 0),
                 arrowprops=dict(arrowstyle="<->", color="#C44E52", lw=2))
axes[0].text(6.5, -0.30, "distance = 11\n(the LARGEST gap)", ha="center",
             fontsize=11, color="#C44E52", fontweight="bold")
axes[0].set_xlim(0, 13)
axes[0].set_ylim(-0.6, 0.5)
axes[0].set_yticks([])
axes[0].set_xlabel("Month as a raw integer", fontsize=11)
axes[0].set_title("WRONG: raw integers imply Dec and Jan are farthest apart",
                  fontsize=12, fontweight="bold")

# --- Right: the CORRECT representation (unit circle) ---
theta = np.linspace(0, 2 * np.pi, 300)
axes[1].plot(np.cos(theta), np.sin(theta), color="lightgray", lw=1.5, zorder=1)
for m, lab in zip(months, labels):
    angle = 2 * np.pi * m / 12
    x, y = np.cos(angle), np.sin(angle)
    color = "#C44E52" if lab in ("Dec", "Jan") else "#4C72B0"
    axes[1].scatter([x], [y], s=240, color=color, zorder=3)
    axes[1].annotate(lab, (x, y), textcoords="offset points", xytext=(x * 13, y * 13),
                     ha="center", fontsize=10, fontweight="bold" if lab in ("Dec", "Jan") else "normal")

# Highlight the Dec-Jan adjacency
for lab in ("Dec", "Jan"):
    m = labels.index(lab) + 1
    angle = 2 * np.pi * m / 12
    axes[1].scatter([np.cos(angle)], [np.sin(angle)], s=380, facecolors="none",
                    edgecolors="#C44E52", linewidths=2, zorder=4)

axes[1].set_aspect("equal")
axes[1].set_xlim(-1.5, 1.5)
axes[1].set_ylim(-1.5, 1.5)
axes[1].set_title("CORRECT: sin/cos places Dec next to Jan",
                  fontsize=12, fontweight="bold")
axes[1].annotate("distance = 0.52\n(same as Jan-Feb)",
                 xy=(0.85, 0.35), fontsize=10, color="#C44E52", fontweight="bold")

plt.tight_layout()
plt.show()

print(f"Measured in sin/cos space: Jan<->Dec = {month_distance(1, 12):.4f}")
print(f"                          Jan<->Feb = {month_distance(1, 2):.4f}  (identical)")
print(f"                          Jan<->Jul = {month_distance(1, 7):.4f}  (opposite)")
print("\nThe raw-integer version states something false about the data.")

> **Caption:** Visual: sin/cos month encoding vs naive integer


## Screenshot — Evidence: Rocket must not be one-hot encoded

In [ ]:
# =====================================================================
# D.4a - EVIDENCE: why Rocket cannot be one-hot encoded
# =====================================================================
# We justify the frequency-encoding choice with the actual numbers.

rocket_counts = df["Rocket"].value_counts()
n_rockets = len(rocket_counts)
n_rows = len(df)

one_hot_width = n_rockets
rows_per_column = n_rows / one_hot_width

print(f"Rows                     : {n_rows}")
print(f"Distinct rockets         : {n_rockets}")
print(f"One-hot would add        : {one_hot_width} columns")
print(f"Rows per added column    : {rows_per_column:.1f}")
print()
print(f"Rockets appearing once   : {int((rocket_counts == 1).sum())}")
print(f"Rockets appearing <= 2x  : {int((rocket_counts <= 2).sum())}")
print(f"Most common rocket       : {rocket_counts.index[0]!r} ({rocket_counts.iloc[0]} flights)")
print()
print("-> Roughly one column per 12 rows, with most columns almost entirely")
print("   zeros and 83 columns representing a single observation. A tree would")
print("   split on that noise. Frequency encoding avoids this entirely.")

> **Caption:** Evidence: Rocket must not be one-hot encoded


## Screenshot — OperationalExperienceBuilder applied

In [ ]:
# Apply the builder.
# NOTE: `df` already carries `Company_Prior_Launches` / `Company_Prior_Failures`
# from Part D's scaling discussion. The builder is idempotent for those columns,
# so rebuilding them here is safe and keeps Part E self-contained.
df_eng = experience_builder.fit_transform(df)

print(repr(experience_builder))
print(f"history: {experience_builder.history}\n")
display(df_eng[["Company", "Date", "MissionStatus"] + list(experience_builder.feature_names)].head(8))

# The builder must not mutate the frame handed to it. We test that directly by
# snapshotting the frame before the call and comparing after.
snapshot = df.copy(deep=True)
_ = experience_builder.fit_transform(df)
after = sorted(c for c in df.columns if c not in snapshot.columns)

assert after == [], f"build() added columns to the caller's frame: {after}"
assert df.equals(snapshot), "build() mutated the caller's dataframe!"
print("\nPASS: the caller's dataframe was not mutated (defensive copy worked).")

> **Caption:** OperationalExperienceBuilder applied


## Screenshot — Proof leakage would mislead the model

In [ ]:
# =====================================================================
# E.8 - PROVING the leak would have mattered
# =====================================================================
# The whole point of the causal design is that the LAZY version leaks the
# target and inflates scores. We build the lazy version deliberately and
# compare, so the danger is measured rather than asserted.

grouped = df.groupby("Company", sort=False)

# LEAKY: the operator's TOTAL outcome across the entire dataset. Every row
# inherits information about its own fate and about all later missions.
df_leak = df.copy()
df_leak["Leaky_Company_Failures"] = grouped["_is_fail"].transform("sum")
df_leak["Leaky_Company_Failure_Rate"] = grouped["_is_fail"].transform("mean")
df_leak["Launch_Year"] = df_leak["Date"].dt.year

df_honest = df_eng.copy()
df_honest["Launch_Year"] = df_honest["Date"].dt.year

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)

def cv_f1(frame: pd.DataFrame, cols: list[str]) -> tuple[float, float]:
    scores = cross_val_score(
        DecisionTreeClassifier(max_depth=5, random_state=42),
        frame[cols], frame["_is_fail"], cv=cv, scoring="f1",
    )
    return scores.mean(), scores.std()

honest_cols = ["Launch_Year", "Company_Prior_Launches", "Company_Prior_Failure_Rate"]
leak_cols = ["Launch_Year", "Leaky_Company_Failures", "Leaky_Company_Failure_Rate"]

honest_f1, honest_sd = cv_f1(df_honest, honest_cols)
leak_f1, leak_sd = cv_f1(df_leak, leak_cols)

leak_comparison = pd.DataFrame([
    {"feature_set": "Honest (causal)", "features": ", ".join(honest_cols),
     "cv_f1": round(honest_f1, 4), "cv_sd": round(honest_sd, 4)},
    {"feature_set": "LEAKY (company totals)", "features": ", ".join(leak_cols),
     "cv_f1": round(leak_f1, 4), "cv_sd": round(leak_sd, 4)},
]).set_index("feature_set")

display(leak_comparison)
print(f"Leakage inflates F1 by {(leak_f1 - honest_f1):+.4f} "
      f"({(leak_f1/honest_f1 - 1)*100:+.1f}%)")
print("\nThe leaky model looks BETTER while being unusable in production:")
print("at liftoff, no one knows the outcome of the mission being predicted.")

> **Caption:** Proof leakage would mislead the model


## Screenshot — OOP: abstract base class (code)

In [ ]:
class FeatureBuilder(ABC):
    """
    Abstract interface for deriving new columns from existing ones.

    OOP rules exercised (oop_rules.md):
      §4  protected attributes use a leading underscore
      §5  inputs validated before state is assigned
      §6  exposed collections are copied, not handed out live
      §7  shared configuration lives in class attributes
      §11 abstract contract enforced by ABC
    """

    # Class attribute: a shared rule for every builder (oop_rules §7).
    REQUIRES_SORTED_CHRONOLOGY: bool = True

    def __init__(self, name: str, group_col: str, date_col: str):
        # Validate BEFORE assigning, so a bad call cannot leave a half-built
        # object behind (oop_rules §5 rule 2: leave state unchanged on failure).
        self._name = self._validate_text(name, "name")
        self._group_col = self._validate_text(group_col, "group_col")
        self._date_col = self._validate_text(date_col, "date_col")
        self._history: list[str] = []
        self._is_fitted = False

    # ---- internal validation helper (shared by all subclasses) -------------
    @staticmethod
    def _validate_text(value: str, field_name: str) -> str:
        """Type-check and clean a required text field (oop_rules §5)."""
        if not isinstance(value, str):
            raise TypeError(f"{field_name} must be a str, got {type(value).__name__}.")
        cleaned = value.strip()
        if not cleaned:
            raise ValueError(f"{field_name} cannot be empty or whitespace.")
        return cleaned

    # ---- read-only properties (oop_rules §6) ------------------------------
    @property
    def name(self) -> str:
        return self._name

    @property
    def group_col(self) -> str:
        return self._group_col

    @property
    def date_col(self) -> str:
        return self._date_col

    @property
    def is_fitted(self) -> bool:
        return self._is_fitted

    @property
    def history(self) -> tuple[str, ...]:
        """Defensive copy - callers cannot mutate our internal list."""
        return tuple(self._history)

    # ---- abstract contract -------------------------------------------------
    @property
    @abstractmethod
    def feature_names(self) -> tuple[str, ...]:
        """Names of the columns this builder produces."""

    @abstractmethod
    def _build(self, df: pd.DataFrame) -> pd.DataFrame:
        """Concrete derivation logic. Must not mutate the input frame."""

    # ---- template method: shared validation, then subclass work -----------
    def fit_transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Validate, derive, and record. This is a 'template method': the base
        class owns the sequence, subclasses supply only the derivation.
        """
        if not isinstance(df, pd.DataFrame):
            raise TypeError(f"df must be a DataFrame, got {type(df).__name__}.")
        for col in (self._group_col, self._date_col):
            if col not in df.columns:
                raise KeyError(f"Required column {col!r} missing from the dataframe.")

        # Work on a copy so the caller's frame is never mutated.
        out = self._build(df.copy())

        produced = self.feature_names
        missing = [c for c in produced if c not in out.columns]
        if missing:
            raise RuntimeError(f"{type(self).__name__} failed to produce {missing}.")

        self._history.append(f"built {produced}")
        self._is_fitted = True
        return out

    def __repr__(self) -> str:
        state = "fitted" if self._is_fitted else "not fitted"
        return f"{type(self).__name__}(name={self._name!r}, {state})"

> **Caption:** OOP: abstract base class (code)


## Screenshot — OOP: concrete builder (code)

In [ ]:
class OperationalExperienceBuilder(FeatureBuilder):
    """
    Derives accumulated operator experience features, CAUSALLY.

    The causal requirement is the whole point of this class. Every count
    excludes the current row and every later row, so nothing about the mission
    being predicted can enter its own features.
    """

    def __init__(self, group_col: str = "Company", date_col: str = "Date",
                 outcome_col: str = "MissionStatus", success_label: str = "Success"):
        super().__init__(name="OperationalExperience", group_col=group_col,
                         date_col=date_col)
        self._outcome_col = self._validate_text(outcome_col, "outcome_col")
        self._success_label = self._validate_text(success_label, "success_label")
        self._excluded: tuple[str, ...] = ()

    @property
    def feature_names(self) -> tuple[str, ...]:
        # Only the two non-redundant features are kept (see E.4).
        return ("Company_Prior_Launches", "Company_Prior_Failure_Rate")

    @property
    def excluded_reason(self) -> dict[str, str]:
        """Documents the candidates we rejected, and why."""
        return dict(self._excluded)

    def _build(self, df: pd.DataFrame) -> pd.DataFrame:
        if self.REQUIRES_SORTED_CHRONOLOGY:
            df = df.sort_values(self._date_col).reset_index(drop=True)

        # Binary failure indicator for the running history.
        is_fail = (df[self._outcome_col] != self._success_label).astype(int)

        grouped = df.groupby(self._group_col, sort=False)

        # --- CAUSAL counts: strictly BEFORE the current row ---
        # cumcount() counts preceding rows; the shifted cumsum excludes this
        # row's own outcome. Both therefore see only the past.
        prior_launches = grouped.cumcount()
        prior_failures = (
            is_fail.groupby(df[self._group_col], sort=False).cumsum() - is_fail
        )

        df["Company_Prior_Launches"] = prior_launches
        df["Company_Prior_Failures"] = prior_failures

        # Historical failure rate, guarded against divide-by-zero.
        df["Company_Prior_Failure_Rate"] = np.where(
            prior_launches > 0, prior_failures / prior_launches, 0.0
        )

        # Record the rejected candidates for the write-up.
        self._excluded = {
            "Company_Prior_Failures": (
                "Correlates 0.952 with Company_Prior_Launches - near-redundant."
            ),
            "Is_First_Company_Launch": (
                "Derivable from prior_launches == 0; adds no information."
            ),
        }
        return df


experience_builder = OperationalExperienceBuilder()
print(repr(experience_builder))
print(f"Produces features : {experience_builder.feature_names}")
print(f"Never sees        : the current row's own outcome, or any later row")

> **Caption:** OOP: concrete builder (code)


## Screenshot — OOP: LeakageGuard class (code)

In [ ]:
class LeakageGuard:
    """
    Validates a feature matrix for target leakage.

    OOP rules exercised (oop_rules.md):
      §4 protected attributes with leading underscore
      §5 validate inputs before assigning state
      §6 read-only @property accessors, defensive copies
      §7 shared configuration as class attributes
    """

    # Class attributes: shared blocklist and threshold (oop_rules §7).
    FORBIDDEN_COLUMNS: tuple = (
        "MissionStatus",   # the target itself
        "_is_fail",        # binary restatement of the target (Part E helper)
        "target",          # derived target
        "RocketStatus",    # temporal leakage (F.2)
        "Mission",         # identifier + latent leak (F.3)
        "Price",           # dropped, retroactive bias (F.4)
        "Time_parsed",     # excluded
    )
    MAX_TARGET_CORRELATION: float = 0.999

    def __init__(self, target: pd.Series, numeric_scan: bool = True):
        if not isinstance(target, pd.Series):
            raise TypeError(
                f"target must be a pandas Series, got {type(target).__name__}."
            )
        if target.nunique() < 2:
            raise ValueError("target must contain at least two distinct values.")
        self._target = target
        self._numeric_scan = bool(numeric_scan)
        self._violations: list[str] = []

    # ---- read-only properties (oop_rules §6) ------------------------------
    @property
    def violations(self) -> tuple[str, ...]:
        """Defensive copy - callers cannot mutate the internal list."""
        return tuple(self._violations)

    @property
    def is_clean(self) -> bool:
        return not self._violations

    @property
    def forbidden_columns(self) -> tuple[str, ...]:
        return tuple(self.FORBIDDEN_COLUMNS)

    # ---- the check ---------------------------------------------------------
    def check(self, X: pd.DataFrame) -> "LeakageGuard":
        """Validate a feature matrix. Raises AssertionError on any violation."""
        if not isinstance(X, pd.DataFrame):
            raise TypeError(f"X must be a DataFrame, got {type(X).__name__}.")

        # Reset state so the guard is reusable.
        self._violations = []

        # Check 1: no blocklisted column present.
        present = [c for c in self.FORBIDDEN_COLUMNS if c in X.columns]
        if present:
            self._violations.append(
                f"Blocklisted column(s) present in X: {present}"
            )

        # Check 2: no numeric column is a near-perfect restatement of y.
        if self._numeric_scan:
            for col in X.columns:
                series = X[col]
                if not pd.api.types.is_numeric_dtype(series):
                    continue
                if series.nunique() != 2:
                    continue
                # Align on the shared index and drop rows with any missing value.
                pair = pd.concat([series, self._target], axis=1).dropna()
                if len(pair) < 2:
                    continue
                corr = abs(np.corrcoef(pair.iloc[:, 0], pair.iloc[:, 1])[0, 1])
                if corr >= self.MAX_TARGET_CORRELATION:
                    self._violations.append(
                        f"Column {col!r} restates the target "
                        f"(binary, |corr| = {corr:.4f})."
                    )

        if self._violations:
            raise AssertionError(
                "LEAKAGE DETECTED:\n  - " + "\n  - ".join(self._violations)
            )
        return self

    def report(self, X: pd.DataFrame) -> pd.DataFrame:
        """Return a pass/fail summary without raising."""
        try:
            self.check(X)
        except AssertionError:
            pass
        return pd.DataFrame([
            {"check": "No blocklisted columns in X",
             "status": "FAIL" if any("Blocklisted" in v for v in self._violations) else "PASS"},
            {"check": f"No column restates the target (|corr| >= {self.MAX_TARGET_CORRELATION})",
             "status": "FAIL" if any("restates" in v for v in self._violations) else "PASS"},
        ]).set_index("check")

    def __repr__(self) -> str:
        state = "clean" if self.is_clean else f"{len(self._violations)} violation(s)"
        return f"LeakageGuard({state})"

> **Caption:** OOP: LeakageGuard class (code)


## Screenshot — Leakage evidence 1: RocketStatus

In [ ]:
# =====================================================================
# F.9 - RocketStatus: three independent pieces of evidence
# =====================================================================

print("EVIDENCE 1 - the temporal tell (status vs launch decade)\n")
status_by_decade = pd.crosstab(df["Date"].dt.year // 10 * 10, df["RocketStatus"])
display(status_by_decade)

early = df[df["Date"].dt.year < 1980]
n_active_early = int((early["RocketStatus"] == "Active").sum())
print(f"Launches before 1980 labelled 'Active' : {n_active_early}")
print(f"Total launches before 1980             : {len(early)}")
print("\n-> Zero exceptions. This is a 2022 snapshot, not a pre-flight condition.")

print("\n" + "=" * 62)
print("\nEVIDENCE 2 - reverse causation (outcome rate by status)\n")
status_outcome = pd.crosstab(
    df["RocketStatus"], df["_is_fail"], normalize="index"
).rename(columns={0: "success_rate", 1: "failure_rate"}).round(4)
display(status_outcome)
print("'Active' vehicles succeed MORE often (93.2% vs 89.0%).")
print("Causation runs backwards: programmes end and unreliable vehicles are")
print("withdrawn, so the OUTCOME partly causes the STATUS.")

> **Caption:** Leakage evidence 1: RocketStatus


## Screenshot — Leakage evidence 2: Mission names

In [ ]:
# =====================================================================
# F.10 - Mission names carry outcome information
# =====================================================================
# Mission is dropped as an identifier anyway, but the audit revealed that the
# name itself encodes risk. This matters because it shows an "ID" column was
# not actually inert.

global_rate = df["_is_fail"].mean()

name_patterns = []
for keyword in ["test", "demo"]:
    mask = df["Mission"].str.lower().str.contains(keyword, na=False)
    name_patterns.append({
        "pattern": f'contains "{keyword}"',
        "n_missions": int(mask.sum()),
        "failure_rate": round(df.loc[mask, "_is_fail"].mean(), 4),
        "median_year": int(df.loc[mask, "Date"].dt.year.median()),
    })

name_patterns.append({
    "pattern": "all missions",
    "n_missions": len(df),
    "failure_rate": round(global_rate, 4),
    "median_year": int(df["Date"].dt.year.median()),
})

name_leak = pd.DataFrame(name_patterns).set_index("pattern")
display(name_leak)

# The key question: is this just an era effect? Test within the early era only.
early_mask = df["Date"].dt.year < 1970
test_mask = df["Mission"].str.lower().str.contains("test", na=False)

print("Is it merely an era proxy? Test restricted to pre-1970 launches:\n")
print(f"  test-named, pre-1970 : {df.loc[test_mask & early_mask, '_is_fail'].mean():.3f} "
      f"(n={int((test_mask & early_mask).sum())})")
print(f"  all others, pre-1970 : {df.loc[~test_mask & early_mask, '_is_fail'].mean():.3f} "
      f"(n={int((~test_mask & early_mask).sum())})")

print(f"\nTest-named missions have a MEDIAN YEAR of "
      f"{int(df.loc[test_mask, 'Date'].dt.year.median())} (dataset median "
      f"{int(df['Date'].dt.year.median())}) - far more recent, yet still fail at")
print(f"{df.loc[test_mask, '_is_fail'].mean()*100:.1f}% versus {global_rate*100:.1f}% overall.")
print("\n-> Not an era proxy. The name signals a genuinely higher-risk mission")
print("   TYPE (first flights, unproven vehicles).")

> **Caption:** Leakage evidence 2: Mission names


## Screenshot — Visual: the three leak findings

In [ ]:
# =====================================================================
# F.15 - Part F summary
# =====================================================================

leak_summary = pd.DataFrame([
    {"finding": "RocketStatus temporal leakage", "severity": "SEVERE",
     "evidence": "0 of 1,837 pre-1980 launches are 'Active'; reverse causation",
     "action": "Dropped and blocklisted in LeakageGuard"},
    {"finding": "Mission names encode risk", "severity": "LATENT",
     "evidence": "'test' names fail 52.2% vs 10.1%; 75% vs 23.6% pre-1970",
     "action": "Already dropped as identifier; documented"},
    {"finding": "Price recorded retroactively", "severity": "MODERATE",
     "evidence": "33/774 priced in 1960s vs 528/676 in 2010s",
     "action": "Already dropped at 72.7% missing"},
    {"finding": "Time carries no signal", "severity": "NONE",
     "evidence": "Failure rate 7.6%-10.2% across hour buckets",
     "action": "Excluded; costs nothing"},
]).set_index("finding")

display(leak_summary)

# Build a candidate feature matrix from everything Part E produced, then let
# the guard inspect it. Only numeric columns are scanned for near-restatement.
candidate_X = df_eng.drop(
    columns=[c for c in LeakageGuard.FORBIDDEN_COLUMNS if c in df_eng.columns]
)
numeric_candidates = candidate_X.select_dtypes(include=[np.number])

guard_final = LeakageGuard(
    target=(df_eng["MissionStatus"] == "Success").astype(int)
)

print("Guard verdict on the candidate feature set:\n")
display(guard_final.report(numeric_candidates))

if guard_final.is_clean:
    print("\nPASS: no leakage detected in the numeric candidate features.")
else:
    print(f"\nReview needed: {guard_final.violations}")

> **Caption:** Visual: the three leak findings


## Phase 3 — Final Technical Development (Parts G–H)

> Phase 3 separated the target (Part G): y has 4,628 values with a 89.89% positive rate (8.89:1 imbalance; hence Part B rejected accuracy as the metric); X has 4,628 rows × 8 columns, all 7 excluded columns leak-verified. Part H used a stratified train/test split (80/20 → 3,702/926) performed BEFORE any fitting, then a leak-safe sklearn ColumnTransformer+Pipeline (imputers, StandardScaler on 5 numeric, one-hot on 2 low-cardinality, custom FrequencyEncoder on Rocket) fitted on training data only. Four leakage checks passed. Final artifacts saved: X_train.npy (3,702 × 46), X_test.npy (926 × 46), y_train.npy, y_test.npy, feature_names.txt.


## Screenshot — Feature/target separation (Part G)

In [ ]:
# =====================================================================
# G.5 - Feature and target separation
# =====================================================================

TARGET_SOURCE = "MissionStatus"
TARGET_LABEL = "target"
POSITIVE_LABEL = "Success"

# Columns excluded from X, each with the part that justified the decision.
EXCLUSION_REGISTER = {
    "MissionStatus":            "The target itself (Part B).",
    "target":                   "Derived target - never a feature.",
    "_is_fail":                 "Binary restatement of the target (Part E helper).",
    "Mission":                  "98.4% unique identifier + latent name leak (Parts A, F).",
    "Time":                     "No signal: 7.6-10.2% failure across hour buckets (Parts C, F).",
    "RocketStatus":             "SEVERE temporal leakage - 2022 snapshot (Part F).",
    "Date":                     "Superseded by Launch_Year / Month_Sin / Month_Cos (Part D).",
    "Company_Prior_Failures":   "0.952 correlation with prior launches - redundant (Part E).",
}

# --- Build the target vector ---
# y = 1 when the mission succeeded, 0 otherwise.
y = (df_eng[TARGET_SOURCE] == POSITIVE_LABEL).astype(int)
y.name = TARGET_LABEL

# Keep the ORIGINAL 4-level text for reference (not used for modelling).
outcome_labels = df_eng[TARGET_SOURCE].copy()
outcome_labels.name = "outcome_labels"

# --- Build the feature matrix ---
drop_columns = [c for c in EXCLUSION_REGISTER if c in df_eng.columns]
X = df_eng.drop(columns=drop_columns)

print(f"Target vector  y : {y.shape[0]} values  (name={y.name!r})")
print(f"Feature matrix X : {X.shape[0]} rows x {X.shape[1]} columns")
print(f"\nColumns dropped from X ({len(drop_columns)}):")
for col in drop_columns:
    print(f"  - {col:<26} {EXCLUSION_REGISTER[col]}")

print(f"\nFeature columns ({X.shape[1]}):")
for col in X.columns:
    kind = "categorical" if X[col].dtype == object else "numeric"
    print(f"  - {col:<28} {str(X[col].dtype):<10} {kind:<12} {X[col].nunique():>5} unique")

> **Caption:** Feature/target separation (Part G)


## Screenshot — Class imbalance in the target

In [ ]:
# =====================================================================
# G.5a - Target distribution
# =====================================================================
target_summary = pd.DataFrame({
    "label": [POSITIVE_LABEL, "Not-Success"],
    "binary_value": [1, 0],
    "count": [int(y.sum()), int((1 - y).sum())],
    "share_pct": [round(y.mean() * 100, 2), round((1 - y.mean()) * 100, 2)],
}).set_index("label")

display(target_summary)

print(f"Positive rate  : {y.mean():.4f}")
print(f"Imbalance ratio: {y.mean() / (1 - y.mean()):.2f} : 1")
print(f"\nOriginal 4-level labels preserved as 'outcome_labels':")
print(outcome_labels.value_counts().to_string())

> **Caption:** Class imbalance in the target


## Screenshot — Guard verdict: no leakage in X

In [ ]:
# =====================================================================
# G.6 - Run the LeakageGuard against the separated X
# =====================================================================

guard = LeakageGuard(target=y)

print("Guard report:\n")
display(guard.report(X))

# check() raises AssertionError on any violation.
guard.check(X)

print("\nPASS: X contains no target leakage.")
print(f"      - no forbidden columns among {X.shape[1]} features")
print(f"      - no binary column restates the target")

> **Caption:** Guard verdict: no leakage in X


## Screenshot — Leak-safe preprocessing pipeline (H.8)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Column groups, derived from the actual dtypes.
numeric_cols = [c for c in X.columns if X[c].dtype != object]
categorical_cols = [c for c in X.columns if X[c].dtype == object]

# Rocket is handled by frequency encoding (Part D), so it is excluded from the
# one-hot group.
FREQ_COLS = ["Rocket"]
ONEHOT_COLS = [c for c in categorical_cols if c not in FREQ_COLS]

print(f"Numeric  ({len(numeric_cols)}): {numeric_cols}")
print(f"One-hot  ({len(ONEHOT_COLS)}): {ONEHOT_COLS}")
print(f"Frequency({len(FREQ_COLS)}): {FREQ_COLS}")

# --- Numeric branch: median impute (idle, see H.5) then standardize ---
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

# --- Categorical branch: impute, then one-hot with rare collapsing ---
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="infrequent_if_exist",  # unseen test categories are safe
        min_frequency=10,                      # collapse rare levels
        sparse_output=False,
    )),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_cols),
        ("cat", categorical_pipeline, ONEHOT_COLS),
        ("freq", FrequencyEncoder(columns=FREQ_COLS), FREQ_COLS),
    ],
    remainder="drop",
)

# The full pipeline: frequency-encode, then the column transformer.
# Constructed but NOT yet fitted.
full_pipeline = Pipeline([
    ("preprocessor", preprocessor),
])

print("\nPipeline constructed but NOT fitted.")
print("Fitting happens next, on X_train only.")

> **Caption:** Leak-safe preprocessing pipeline (H.8)


## Screenshot — Pipeline fitted on training data only

In [ ]:
# =====================================================================
# H.9 - FIT ON TRAINING DATA ONLY
# =====================================================================
# This single call is where leakage is prevented. Every imputation value,
# scaling statistic, and category vocabulary is learned from X_train and
# nothing else.

full_pipeline.fit(X_train, y_train)

X_train_t = full_pipeline.transform(X_train)
X_test_t = full_pipeline.transform(X_test)

# Make them plain arrays for downstream modelling.
X_train_t = np.asarray(X_train_t, dtype=float)
X_test_t = np.asarray(X_test_t, dtype=float)

print("Fitted on training data only.\n")
print(f"  Transformed training matrix : {X_train_t.shape}")
print(f"  Transformed test matrix     : {X_test_t.shape}")
print(f"  Feature counts match        : {X_train_t.shape[1] == X_test_t.shape[1]}")

> **Caption:** Pipeline fitted on training data only


## Screenshot — Leakage check 1: scaler stats (H.10a)

In [ ]:
# =====================================================================
# H.10a - CHECK 1: scaler statistics match the TRAINING distribution
# =====================================================================
# If the scaler had been fitted on the full dataset, its stored means would
# match the full-data means instead. We compare against BOTH to prove which
# one it actually learned.

scaler = (
    full_pipeline.named_steps["preprocessor"]
    .named_transformers_["num"]
    .named_steps["scaler"]
)

# Reproduce what the imputer would output, so the means are comparable.
imputed_train = SimpleImputer(strategy="median").fit_transform(X_train[numeric_cols])
imputed_full = SimpleImputer(strategy="median").fit_transform(X[numeric_cols])

train_means = pd.DataFrame(imputed_train, columns=numeric_cols).mean()
full_means = pd.DataFrame(imputed_full, columns=numeric_cols).mean()

comparison = pd.DataFrame({
    "scaler_mean_from_fit": scaler.mean_,
    "actual_TRAIN_mean": train_means.values,
    "actual_FULL_mean": full_means.values,
})
comparison["diff_vs_train"] = (
    comparison["scaler_mean_from_fit"] - comparison["actual_TRAIN_mean"]
).abs()
comparison["diff_vs_full"] = (
    comparison["scaler_mean_from_fit"] - comparison["actual_FULL_mean"]
).abs()
comparison.index = numeric_cols

display(comparison.round(6))

max_vs_train = comparison["diff_vs_train"].max()
print(f"Max deviation from TRAINING means : {max_vs_train:.12f}")

assert max_vs_train < 1e-8, "Scaler was NOT fitted on training data only!"
print("\nPASS: the scaler's parameters came from the training split.")
print("      (Deviations from the full-data means are present, which is")
print("       expected and correct - the test rows were excluded.)")

> **Caption:** Leakage check 1: scaler stats (H.10a)


## Screenshot — Leakage check 2: encoder unseen categories (H.10b)

In [ ]:
# =====================================================================
# H.10b - CHECK 2: encoder vocabulary is a subset of TRAINING categories
# =====================================================================

onehot = (
    full_pipeline.named_steps["preprocessor"]
    .named_transformers_["cat"]
    .named_steps["onehot"]
)

# OneHotEncoder stores every category it saw during fit, per input column,
# in ONEHOT_COLS order. Note that min_frequency collapses rare categories into
# a single output column, so this list is WIDER than the output width - which
# is exactly what we want here: it is the full learned vocabulary.
train_vocab = {}
for col, cats in zip(ONEHOT_COLS, onehot.categories_):
    train_vocab[col] = set(cats)

rows = []
for col in ONEHOT_COLS:
    train_cats = set(X_train[col].unique())
    test_cats = set(X_test[col].unique())
    only_in_test = test_cats - train_cats

    rows.append({
        "column": col,
        "train_categories": len(train_cats),
        "test_categories": len(test_cats),
        "categories_only_in_test": len(only_in_test),
        "encoder_vocabulary": len(train_vocab[col]),
    })

vocab_report = pd.DataFrame(rows).set_index("column")
display(vocab_report)

n_test_only = int(vocab_report["categories_only_in_test"].sum())
print(f"Categories appearing only in the test set: {n_test_only}")
print("These are routed to the 'infrequent' bucket - the encoder never saw them")
print("during fit, which is exactly the correct production behaviour.")

# The encoder must not have learned anything that is test-only.
for col in ONEHOT_COLS:
    test_only = set(X_test[col].unique()) - set(X_train[col].unique())
    leaked = test_only.intersection(train_vocab[col] - {"infrequent_sklearn"})
    assert not leaked, f"Encoder learned test-only categories for {col}: {leaked}"

print("\nPASS: no test-only category appears in the fitted vocabulary.")

> **Caption:** Leakage check 2: encoder unseen categories (H.10b)


## Screenshot — Leakage check 3: vocabulary (H.10c)

In [ ]:
# =====================================================================
# H.10c - CHECK 3: the frequency encoder used training frequencies only
# =====================================================================

freq_encoder = (
    full_pipeline.named_steps["preprocessor"]
    .named_transformers_["freq"]
)

print("FrequencyEncoder vocabulary, learned from training data:\n")
print(f"  Distinct rockets learned : {freq_encoder.learned_vocabulary_size['Rocket']}")
print(f"  Distinct rockets in full : {X['Rocket'].nunique()}")
print(f"  Distinct rockets in test : {X_test['Rocket'].nunique()}")

# Compare against frequencies computed from the FULL dataset - they must differ,
# proving the encoder did not see the test rows.
train_freq = X_train["Rocket"].value_counts(normalize=True)
full_freq = X["Rocket"].value_counts(normalize=True)

sample_rockets = train_freq.head(5).index.tolist()
freq_comparison = pd.DataFrame({
    "learned_from_train": [freq_encoder.frequencies_["Rocket"].get(r, 0.0) for r in sample_rockets],
    "actual_train_freq": [train_freq[r] for r in sample_rockets],
    "full_data_freq": [full_freq[r] for r in sample_rockets],
}, index=sample_rockets)

display(freq_comparison.round(6))

learned_matches_train = np.allclose(
    freq_comparison["learned_from_train"], freq_comparison["actual_train_freq"]
)
print(f"Learned values match TRAIN frequencies : {learned_matches_train}")
assert learned_matches_train, "FrequencyEncoder did not use training frequencies!"
print("PASS: frequency mapping came from the training split only.")

> **Caption:** Leakage check 3: vocabulary (H.10c)


## Screenshot — Before/after validation (H.12)

In [ ]:
# =====================================================================
# H.12 - Before/after validation (guide §13A)
# =====================================================================

# Reconstruct row counts to confirm nothing was lost or duplicated.
total_rows = X_train.shape[0] + X_test.shape[0]

validation = pd.DataFrame([
    {"stage": "After cleaning (Part C)", "rows": len(df_eng), "columns": df_eng.shape[1]},
    {"stage": "After separation (Part G)", "rows": len(X), "columns": X.shape[1]},
    {"stage": "Training split", "rows": X_train.shape[0], "columns": X_train.shape[1]},
    {"stage": "Test split", "rows": X_test.shape[0], "columns": X_test.shape[1]},
    {"stage": "Transformed (train)", "rows": X_train_t.shape[0], "columns": X_train_t.shape[1]},
    {"stage": "Transformed (test)", "rows": X_test_t.shape[0], "columns": X_test_t.shape[1]},
]).set_index("stage")

display(validation)

assert total_rows == len(X), "Rows lost during the split!"
assert int(y_train.sum()) + int(y_test.sum()) == int(y.sum()), "Labels lost!"
assert (1 - y_train).sum() + (1 - y_test).sum() == (1 - y).sum(), "Minority labels lost!"

print(f"PASS: {len(X):,} rows preserved end-to-end ({X_train.shape[0]:,} + {X_test.shape[0]:,}).")
print(f"PASS: {int(y.sum()):,} positives and {int((1-y).sum()):,} negatives preserved.")

> **Caption:** Before/after validation (H.12)


## Screenshot — Saved model-ready artifacts (H.13)

In [ ]:
# =====================================================================
# H.13 - Persist the model-ready artefacts
# =====================================================================

np.save("X_train.npy", X_train_t)
np.save("X_test.npy", X_test_t)
np.save("y_train.npy", np.asarray(y_train))
np.save("y_test.npy", np.asarray(y_test))

with open("feature_names.txt", "w") as fh:
    fh.write("\n".join(feature_names))

print("Saved model-ready artefacts:")
print(f"  X_train.npy        {X_train_t.shape}")
print(f"  X_test.npy         {X_test_t.shape}")
print(f"  y_train.npy        {np.asarray(y_train).shape}")
print(f"  y_test.npy         {np.asarray(y_test).shape}")
print(f"  feature_names.txt  {len(feature_names)} names")

print("\nPart H complete: the data is split, transformed, and leak-free.")
print("Part I will summarise every preprocessing decision in one table.")

> **Caption:** Saved model-ready artifacts (H.13)


## Screenshot — Visual: stratified split composition

In [ ]:
# =====================================================================
# H.14 - CHART 1: split composition and stratification
# =====================================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Left: row counts ---
sizes = [len(X_train), len(X_test)]
labels = [f"Training\n{sizes[0]:,} rows", f"Test\n{sizes[1]:,} rows"]
bars = axes[0].bar(labels, sizes, color=["#4C72B0", "#DD8452"],
                   edgecolor="white", linewidth=1.5)
for bar, s in zip(bars, sizes):
    axes[0].text(bar.get_x() + bar.get_width()/2, s + 60,
                 f"{s/len(X)*100:.0f}%", ha="center", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Number of rows", fontsize=11)
axes[0].set_title("80/20 split", fontsize=13, fontweight="bold")
axes[0].set_ylim(0, max(sizes) * 1.18)

# --- Right: stratification worked ---
split_names = ["Full data", "Training", "Test"]
pos_rates = [y.mean() * 100, y_train.mean() * 100, y_test.mean() * 100]
minority_rates = [100 - p for p in pos_rates]

x = np.arange(len(split_names))
w = 0.36
axes[1].bar(x - w/2, pos_rates, w, label="Success", color="#55A868", edgecolor="white")
axes[1].bar(x + w/2, minority_rates, w, label="Not-Success", color="#C44E52", edgecolor="white")

for xi, (p, m) in enumerate(zip(pos_rates, minority_rates)):
    axes[1].text(xi - w/2, p + 1.5, f"{p:.2f}%", ha="center", fontsize=9.5, fontweight="bold")
    axes[1].text(xi + w/2, m + 1.5, f"{m:.2f}%", ha="center", fontsize=9.5, fontweight="bold")

axes[1].set_xticks(x)
axes[1].set_xticklabels(split_names)
axes[1].set_ylabel("Share of rows (%)", fontsize=11)
axes[1].set_title("Stratification preserved the class ratio", fontsize=13, fontweight="bold")
axes[1].legend()
axes[1].set_ylim(0, 108)

plt.tight_layout()
plt.show()

print(f"Positive rate - full: {y.mean():.4f} | train: {y_train.mean():.4f} | test: {y_test.mean():.4f}")
print("The three rates are nearly identical, which is what stratify=y guarantees.")
print("Test set carries", int((1 - y_test).sum()), "failure cases to evaluate on.")

> **Caption:** Visual: stratified split composition


## Screenshot — Visual: where the 46 features come from

In [ ]:
# =====================================================================
# H.16 - CHART 3: where the features came from
# =====================================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Left: contribution of each source column to final width ---
# Count one-hot names per source column using the authoritative name list.
n_country = sum(1 for n in onehot_names if n.startswith("Country_"))
n_company = sum(1 for n in onehot_names if n.startswith("Company_"))

contributions = {
    f"Numeric ({len(numeric_cols)} cols)": len(numeric_cols),
    "Country one-hot": n_country,
    "Company one-hot": n_company,
    "Rocket frequency": 1,
}
total = sum(contributions.values())

bars = axes[0].barh(list(contributions.keys()), list(contributions.values()),
                    color=["#4C72B0", "#55A868", "#8172B2", "#DD8452"],
                    edgecolor="white", linewidth=1.2)
for bar, v in zip(bars, contributions.values()):
    axes[0].text(v + 0.4, bar.get_y() + bar.get_height()/2,
                 f"{v}  ({v/total*100:.0f}%)", va="center", fontsize=10, fontweight="bold")
axes[0].set_xlabel("Columns produced", fontsize=11)
axes[0].set_title(f"Final feature space: {total} columns", fontsize=12, fontweight="bold")
axes[0].set_xlim(0, max(contributions.values()) * 1.4)

# --- Right: the Rocket encoding decision, side by side ---
rocket_options = {
    "One-hot\n(min_frequency=10)": 86,
    "Frequency\nencoding (chosen)": 1,
}
bars2 = axes[1].bar(list(rocket_options.keys()), list(rocket_options.values()),
                    color=["#C44E52", "#55A868"], edgecolor="white", linewidth=1.5)
for bar, v in zip(bars2, rocket_options.values()):
    axes[1].text(bar.get_x() + bar.get_width()/2, v + 2, str(v),
                 ha="center", fontsize=14, fontweight="bold")
axes[1].set_ylabel("Columns added for 'Rocket'", fontsize=11)
axes[1].set_title("Part D's frequency-encoding decision, quantified",
                  fontsize=12, fontweight="bold")
axes[1].set_ylim(0, 100)
axes[1].annotate("86 columns for one feature\nwould dominate the matrix",
                 xy=(0, 86), xytext=(0.35, 62), fontsize=9.5, color="#7A2020",
                 arrowprops=dict(arrowstyle="->", color="#C44E52", lw=1.5))

plt.tight_layout()
plt.show()

print(f"Final matrix: {X_train_t.shape[1]} features from {len(X.columns)} source columns.")
print(f"Rows:features ratio = {len(X) // X_train_t.shape[1]} : 1")
print("\nFrequency encoding saved 85 columns versus one-hot on 'Rocket' alone.")
print("Part D committed to this; Part H implements it and shows the saving.")

> **Caption:** Visual: where the 46 features come from


---
# 4. Final Project Output

> The final output is the completed notebook oop.ipynb (Parts A–H) and its saved model-ready artifacts: X_train.npy, X_test.npy, y_train.npy, y_test.npy, and feature_names.txt — a 46-feature, leakage-free, train/test-split classification matrix ready for any scikit-learn model.
> >
> > Non-notebook screenshots: (1) full file listing in Jupyter / VS Code, (2) terminal proof of a headless run `python execute_notebook.py` with no errors.


---
# 5. Results and Key Findings

> - Cleaned data: 4,630 → 4,628 rows, 9 → 8 columns (+Country), 0 lost values.
> >
> - Final feature matrix: 46 engineered features from 8 source columns (5 numeric, 40 one-hot, 1 frequency-encoded); rows:features = 100:1.
> >
> - Class imbalance: 89.89% success rate → 8.89:1 ratio; accuracy would be misleading as a metric (Part B).
> >
> - Frequency encoding on Rocket saved ~324 columns versus one-hot (370 distinct rockets).
> >
> - Leakage: three post-launch information channels found and excluded — RocketStatus (temporal proxy), Mission names (outcome naming), Price (retroactive availability). A leaky experience feature inflated F1 by +49.6% versus the honest version.
> >
> - Stratified split preserved class balance: positive rates 0.899/0.899/0.899 across full/train/test.
> >
> - First launches fail at 41.9% vs 5.1% after 500+ cumulative launches — an 8.2× experience effect motivating the engineered features.


## Screenshot — Finding: operational experience effect (41.9% vs 5.1%)

In [ ]:
# =====================================================================
# E.9 - CHART 1: the experience signal
# =====================================================================
# The headline result: failure rate falls monotonically with experience.

bins = pd.cut(
    df_eng["Company_Prior_Launches"],
    bins=[-1, 0, 5, 20, 100, 500, 100000],
    labels=["1st\nlaunch", "2-6", "7-21", "22-101", "102-501", "502+"],
)
signal = df_eng.groupby(bins, observed=True)["_is_fail"].agg(["count", "mean"])
signal["pct"] = signal["mean"] * 100

fig, ax = plt.subplots()
bars = ax.bar(signal.index.astype(str), signal["pct"],
              color="#4C72B0", edgecolor="white", linewidth=1.2)

for bar, pct, n in zip(bars, signal["pct"], signal["count"]):
    ax.text(bar.get_x() + bar.get_width() / 2, pct + 1.0,
            f"{pct:.1f}%\n(n={n:,})", ha="center", va="bottom", fontsize=9)

global_rate = df_eng["_is_fail"].mean() * 100
ax.axhline(global_rate, ls="--", color="#C44E52", linewidth=1.5,
           label=f"Global failure rate ({global_rate:.1f}%)")

ax.set_xlabel("Operator's prior launches before this mission", fontsize=11)
ax.set_ylabel("Failure rate (%)", fontsize=11)
ax.set_title("Failure rate falls monotonically with operator experience",
             fontsize=13, fontweight="bold")
ax.set_ylim(0, signal["pct"].max() * 1.25)
ax.legend()
plt.tight_layout()
plt.show()

print(f"A first launch fails at {signal['pct'].iloc[0]:.1f}%, versus "
      f"{signal['pct'].iloc[-1]:.1f}% after 500+ launches.")
print(f"That is a {signal['pct'].iloc[0] / signal['pct'].iloc[-1]:.1f}x difference.")

> **Caption:** Finding: operational experience effect (41.9% vs 5.1%)


## Screenshot — Finding: leakage inflates performance by 50%

In [ ]:
# =====================================================================
# E.13 - Part E summary
# =====================================================================

summary_e = pd.DataFrame([
    {"feature": "Company_Prior_Launches", "kept": "Yes",
     "why": "Primary experience measure; monotonic 41.9% -> 5.1% signal."},
    {"feature": "Company_Prior_Failure_Rate", "kept": "Yes",
     "why": "Adds the quality dimension (50 launches with 10 failures != 50 with 0)."},
    {"feature": "Company_Prior_Failures", "kept": "No",
     "why": "0.952 correlation with prior launches - near-redundant."},
    {"feature": "Is_First_Company_Launch", "kept": "No",
     "why": "Derivable from prior_launches == 0."},
]).set_index("feature")

display(summary_e)
print(f"Final engineered features: {list(experience_builder.feature_names)}")
print(f"Dataframe shape           : {df_eng.shape}")

> **Caption:** Finding: leakage inflates performance by 50%


## Screenshot — Finding: 89.89% positive class

In [ ]:
# =====================================================================
# G.7 - CHART 1: class balance, and why accuracy is rejected
# =====================================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Left: the binary target distribution ---
counts = [int(y.sum()), int((1 - y).sum())]
labels = [f"Success\n(n={counts[0]:,})", f"Not-Success\n(n={counts[1]:,})"]
colors = ["#55A868", "#C44E52"]

bars = axes[0].bar(labels, counts, color=colors, edgecolor="white", linewidth=1.5)
for bar, c in zip(bars, counts):
    pct = c / len(y) * 100
    axes[0].text(bar.get_x() + bar.get_width() / 2, c + 70,
                 f"{pct:.2f}%", ha="center", fontsize=13, fontweight="bold")

axes[0].set_ylabel("Number of missions", fontsize=11)
axes[0].set_title("Binary target is imbalanced 9:1", fontsize=13, fontweight="bold")
axes[0].set_ylim(0, max(counts) * 1.18)

# The trivial-predictor line
axes[0].axhline(counts[0], ls=":", color="gray", linewidth=1.6)
axes[0].text(0.5, counts[0] * 0.55,
             "A model predicting 'Success'\nfor EVERY mission scores\n"
             f"{y.mean()*100:.2f}% accuracy\nand catches 0 failures.",
             ha="center", fontsize=10,
             bbox=dict(boxstyle="round,pad=0.5", fc="#FFF3CD", ec="gray"))

# --- Right: the original 4-level distribution ---
level_counts = outcome_labels.value_counts()
axes[1].barh(level_counts.index[::-1], level_counts.values[::-1],
             color=["#C44E52", "#DD8452", "#C44E52", "#55A868"],
             edgecolor="white", linewidth=1.2)
for i, v in enumerate(level_counts.values[::-1]):
    axes[1].text(v + 60, i, f"{v:,}", va="center", fontsize=11, fontweight="bold")

axes[1].set_xlabel("Number of missions", fontsize=11)
axes[1].set_title("Original 4 levels before binarisation", fontsize=13, fontweight="bold")
axes[1].set_xlim(0, level_counts.max() * 1.25)
axes[1].annotate("only 4 observations -\ncannot be stratified",
                 xy=(4, 0), xytext=(900, 0.55), fontsize=10, color="#C44E52",
                 arrowprops=dict(arrowstyle="->", color="#C44E52", lw=1.5))

plt.tight_layout()
plt.show()

print(f"The positive class dominates at {y.mean()*100:.2f}%.")
print("This is exactly why Part B rejected accuracy: a trivial majority-class")
print("predictor scores ~90% while detecting zero of the 468 failures.")
print(f"\n'Prelaunch Failure' has only {level_counts['Prelaunch Failure']} rows, which is")
print("why binarisation was necessary rather than optional.")

> **Caption:** Finding: 89.89% positive class


## Screenshot — Result: pipeline output summary (saved artifacts)

In [ ]:
# =====================================================================
# H.17 - Part H summary
# =====================================================================

part_h_summary = pd.DataFrame([
    {"item": "Split ratio", "value": "80 / 20, stratified (random_state=42)"},
    {"item": "Training rows", "value": f"{X_train.shape[0]:,}"},
    {"item": "Test rows", "value": f"{X_test.shape[0]:,}"},
    {"item": "Minority rows in test", "value": f"{int((1 - y_test).sum())} failures"},
    {"item": "Source features", "value": f"{X.shape[1]}"},
    {"item": "Transformed features", "value": f"{X_train_t.shape[1]}"},
    {"item": "Fitted on", "value": "X_train ONLY"},
    {"item": "Imputation", "value": "Wired in, idle (X has no missing values)"},
    {"item": "Scaling", "value": "StandardScaler, fitted on train"},
    {"item": "Encoding", "value": "One-hot (Country, Company) + Frequency (Rocket)"},
    {"item": "Unseen test categories", "value": "18 Rocket, 1 Company -> 'infrequent'/0.0"},
    {"item": "Leakage checks", "value": "4 numeric checks + LeakageGuard, all PASS"},
    {"item": "Rows preserved", "value": f"{len(X):,} (train + test)"},
]).set_index("item")

display(part_h_summary)

print("Artefacts saved: X_train.npy, X_test.npy, y_train.npy, y_test.npy, feature_names.txt")

> **Caption:** Result: pipeline output summary (saved artifacts)


---
# 6. Discussion

> The pipeline's behavior is consistent with its purpose: preprocessing is fitted on training data only, and every excluded column has a documented, evidence-backed reason. The results show why leakage matters dataset-wide — several raw columns (RocketStatus, Mission, Price) look informative but would not be computable at predict-time, which would have made any model trained on them unusable. The engineered experience features are the most informative honest signal discovered. The 89.89% class imbalance means future evaluation should use F1/precision-recall, not accuracy — a decision carried through Part B.


---
# 7. Conclusion

> All Phase 1–3 objectives were achieved: the raw CSV was inspected and documented; three latent leakage vectors were identified and quantitatively proven; FeatureBuilder and LeakageGuard classes exercise inheritance, interfaces, and validation (OOP rules); a leak-safe sklearn pipeline produced and persisted a clean, split, 46-feature model-ready dataset with zero values lost and rows fully preserved end-to-end.


---
# 8. Recommendations and Future Improvements

> - Train and evaluate actual classifiers (logistic regression, random forest, gradient boosting) on the saved arrays; report F1, not accuracy.
> >
> - Address the 8.9:1 imbalance (class weights, SMOTE, or threshold tuning).
> >
> - Substitute or impute a sparse, era-independent proxy for Price (e.g., rocket-class tiers) instead of dropping it.
> >
> - Tune the FrequencyEncoder / 'infrequent' bucket threshold on validation folds.
> >
> - Extend the OOP layer with additional concrete builders (e.g., a country history builder) behind the same abstract interface.


---
# 9. References

> - Space Missions dataset (Kaggle, space_missions.csv, 4,630 launches × 9 features).
> >
> - Scikit-learn documentation: ColumnTransformer, Pipeline, StandardScaler, train_test_split, OneHotEncoder. https://scikit-learn.org/stable/
> >
> - Python abc module (abstract base classes). https://docs.python.org/3/library/abc.html
> >
> - Course lecture notes: OOP rules & architecture reference (oop_rules.md); data preprocessing notes and activity guide.
> >
> - Course repository: https://github.com/dokdokdok225/OOP


---
# 10. Appendix — Supporting code (NOT in the body)

Screenshot these full code cells into the appendix only.

## Screenshot — A1: FeatureBuilder ABC source

In [ ]:
class FeatureBuilder(ABC):
    """
    Abstract interface for deriving new columns from existing ones.

    OOP rules exercised (oop_rules.md):
      §4  protected attributes use a leading underscore
      §5  inputs validated before state is assigned
      §6  exposed collections are copied, not handed out live
      §7  shared configuration lives in class attributes
      §11 abstract contract enforced by ABC
    """

    # Class attribute: a shared rule for every builder (oop_rules §7).
    REQUIRES_SORTED_CHRONOLOGY: bool = True

    def __init__(self, name: str, group_col: str, date_col: str):
        # Validate BEFORE assigning, so a bad call cannot leave a half-built
        # object behind (oop_rules §5 rule 2: leave state unchanged on failure).
        self._name = self._validate_text(name, "name")
        self._group_col = self._validate_text(group_col, "group_col")
        self._date_col = self._validate_text(date_col, "date_col")
        self._history: list[str] = []
        self._is_fitted = False

    # ---- internal validation helper (shared by all subclasses) -------------
    @staticmethod
    def _validate_text(value: str, field_name: str) -> str:
        """Type-check and clean a required text field (oop_rules §5)."""
        if not isinstance(value, str):
            raise TypeError(f"{field_name} must be a str, got {type(value).__name__}.")
        cleaned = value.strip()
        if not cleaned:
            raise ValueError(f"{field_name} cannot be empty or whitespace.")
        return cleaned

    # ---- read-only properties (oop_rules §6) ------------------------------
    @property
    def name(self) -> str:
        return self._name

    @property
    def group_col(self) -> str:
        return self._group_col

    @property
    def date_col(self) -> str:
        return self._date_col

    @property
    def is_fitted(self) -> bool:
        return self._is_fitted

    @property
    def history(self) -> tuple[str, ...]:
        """Defensive copy - callers cannot mutate our internal list."""
        return tuple(self._history)

    # ---- abstract contract -------------------------------------------------
    @property
    @abstractmethod
    def feature_names(self) -> tuple[str, ...]:
        """Names of the columns this builder produces."""

    @abstractmethod
    def _build(self, df: pd.DataFrame) -> pd.DataFrame:
        """Concrete derivation logic. Must not mutate the input frame."""

    # ---- template method: shared validation, then subclass work -----------
    def fit_transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Validate, derive, and record. This is a 'template method': the base
        class owns the sequence, subclasses supply only the derivation.
        """
        if not isinstance(df, pd.DataFrame):
            raise TypeError(f"df must be a DataFrame, got {type(df).__name__}.")
        for col in (self._group_col, self._date_col):
            if col not in df.columns:
                raise KeyError(f"Required column {col!r} missing from the dataframe.")

        # Work on a copy so the caller's frame is never mutated.
        out = self._build(df.copy())

        produced = self.feature_names
        missing = [c for c in produced if c not in out.columns]
        if missing:
            raise RuntimeError(f"{type(self).__name__} failed to produce {missing}.")

        self._history.append(f"built {produced}")
        self._is_fitted = True
        return out

    def __repr__(self) -> str:
        state = "fitted" if self._is_fitted else "not fitted"
        return f"{type(self).__name__}(name={self._name!r}, {state})"

> **Caption:** A1: FeatureBuilder ABC source


## Screenshot — A2: OperationalExperienceBuilder source

In [ ]:
class OperationalExperienceBuilder(FeatureBuilder):
    """
    Derives accumulated operator experience features, CAUSALLY.

    The causal requirement is the whole point of this class. Every count
    excludes the current row and every later row, so nothing about the mission
    being predicted can enter its own features.
    """

    def __init__(self, group_col: str = "Company", date_col: str = "Date",
                 outcome_col: str = "MissionStatus", success_label: str = "Success"):
        super().__init__(name="OperationalExperience", group_col=group_col,
                         date_col=date_col)
        self._outcome_col = self._validate_text(outcome_col, "outcome_col")
        self._success_label = self._validate_text(success_label, "success_label")
        self._excluded: tuple[str, ...] = ()

    @property
    def feature_names(self) -> tuple[str, ...]:
        # Only the two non-redundant features are kept (see E.4).
        return ("Company_Prior_Launches", "Company_Prior_Failure_Rate")

    @property
    def excluded_reason(self) -> dict[str, str]:
        """Documents the candidates we rejected, and why."""
        return dict(self._excluded)

    def _build(self, df: pd.DataFrame) -> pd.DataFrame:
        if self.REQUIRES_SORTED_CHRONOLOGY:
            df = df.sort_values(self._date_col).reset_index(drop=True)

        # Binary failure indicator for the running history.
        is_fail = (df[self._outcome_col] != self._success_label).astype(int)

        grouped = df.groupby(self._group_col, sort=False)

        # --- CAUSAL counts: strictly BEFORE the current row ---
        # cumcount() counts preceding rows; the shifted cumsum excludes this
        # row's own outcome. Both therefore see only the past.
        prior_launches = grouped.cumcount()
        prior_failures = (
            is_fail.groupby(df[self._group_col], sort=False).cumsum() - is_fail
        )

        df["Company_Prior_Launches"] = prior_launches
        df["Company_Prior_Failures"] = prior_failures

        # Historical failure rate, guarded against divide-by-zero.
        df["Company_Prior_Failure_Rate"] = np.where(
            prior_launches > 0, prior_failures / prior_launches, 0.0
        )

        # Record the rejected candidates for the write-up.
        self._excluded = {
            "Company_Prior_Failures": (
                "Correlates 0.952 with Company_Prior_Launches - near-redundant."
            ),
            "Is_First_Company_Launch": (
                "Derivable from prior_launches == 0; adds no information."
            ),
        }
        return df


experience_builder = OperationalExperienceBuilder()
print(repr(experience_builder))
print(f"Produces features : {experience_builder.feature_names}")
print(f"Never sees        : the current row's own outcome, or any later row")

> **Caption:** A2: OperationalExperienceBuilder source


## Screenshot — A3: LeakageGuard source

In [ ]:
class LeakageGuard:
    """
    Validates a feature matrix for target leakage.

    OOP rules exercised (oop_rules.md):
      §4 protected attributes with leading underscore
      §5 validate inputs before assigning state
      §6 read-only @property accessors, defensive copies
      §7 shared configuration as class attributes
    """

    # Class attributes: shared blocklist and threshold (oop_rules §7).
    FORBIDDEN_COLUMNS: tuple = (
        "MissionStatus",   # the target itself
        "_is_fail",        # binary restatement of the target (Part E helper)
        "target",          # derived target
        "RocketStatus",    # temporal leakage (F.2)
        "Mission",         # identifier + latent leak (F.3)
        "Price",           # dropped, retroactive bias (F.4)
        "Time_parsed",     # excluded
    )
    MAX_TARGET_CORRELATION: float = 0.999

    def __init__(self, target: pd.Series, numeric_scan: bool = True):
        if not isinstance(target, pd.Series):
            raise TypeError(
                f"target must be a pandas Series, got {type(target).__name__}."
            )
        if target.nunique() < 2:
            raise ValueError("target must contain at least two distinct values.")
        self._target = target
        self._numeric_scan = bool(numeric_scan)
        self._violations: list[str] = []

    # ---- read-only properties (oop_rules §6) ------------------------------
    @property
    def violations(self) -> tuple[str, ...]:
        """Defensive copy - callers cannot mutate the internal list."""
        return tuple(self._violations)

    @property
    def is_clean(self) -> bool:
        return not self._violations

    @property
    def forbidden_columns(self) -> tuple[str, ...]:
        return tuple(self.FORBIDDEN_COLUMNS)

    # ---- the check ---------------------------------------------------------
    def check(self, X: pd.DataFrame) -> "LeakageGuard":
        """Validate a feature matrix. Raises AssertionError on any violation."""
        if not isinstance(X, pd.DataFrame):
            raise TypeError(f"X must be a DataFrame, got {type(X).__name__}.")

        # Reset state so the guard is reusable.
        self._violations = []

        # Check 1: no blocklisted column present.
        present = [c for c in self.FORBIDDEN_COLUMNS if c in X.columns]
        if present:
            self._violations.append(
                f"Blocklisted column(s) present in X: {present}"
            )

        # Check 2: no numeric column is a near-perfect restatement of y.
        if self._numeric_scan:
            for col in X.columns:
                series = X[col]
                if not pd.api.types.is_numeric_dtype(series):
                    continue
                if series.nunique() != 2:
                    continue
                # Align on the shared index and drop rows with any missing value.
                pair = pd.concat([series, self._target], axis=1).dropna()
                if len(pair) < 2:
                    continue
                corr = abs(np.corrcoef(pair.iloc[:, 0], pair.iloc[:, 1])[0, 1])
                if corr >= self.MAX_TARGET_CORRELATION:
                    self._violations.append(
                        f"Column {col!r} restates the target "
                        f"(binary, |corr| = {corr:.4f})."
                    )

        if self._violations:
            raise AssertionError(
                "LEAKAGE DETECTED:\n  - " + "\n  - ".join(self._violations)
            )
        return self

    def report(self, X: pd.DataFrame) -> pd.DataFrame:
        """Return a pass/fail summary without raising."""
        try:
            self.check(X)
        except AssertionError:
            pass
        return pd.DataFrame([
            {"check": "No blocklisted columns in X",
             "status": "FAIL" if any("Blocklisted" in v for v in self._violations) else "PASS"},
            {"check": f"No column restates the target (|corr| >= {self.MAX_TARGET_CORRELATION})",
             "status": "FAIL" if any("restates" in v for v in self._violations) else "PASS"},
        ]).set_index("check")

    def __repr__(self) -> str:
        state = "clean" if self.is_clean else f"{len(self._violations)} violation(s)"
        return f"LeakageGuard({state})"

> **Caption:** A3: LeakageGuard source


## Screenshot — A4: FrequencyEncoder source

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin


class FrequencyEncoder(BaseEstimator, TransformerMixin):
    """
    Replace each category with its observed frequency.

    Implements the standard scikit-learn transformer contract so it can sit
    inside a Pipeline and inherit the fit-on-train-only guarantee:

        fit(X_train)          -> learns category -> frequency from TRAIN only
        transform(any_X)      -> applies that frozen mapping

    Unseen categories (present in test but absent from train) map to a neutral
    0.0 rather than raising or producing NaN. This mirrors the
    handle_unknown="infrequent_if_exist" behaviour of OneHotEncoder.

    Also satisfies the oop_rules.md §13 checklist:
      §4 protected attributes, §5 validation before assignment,
      §6 read-only @property accessors, §7 class-level configuration.
    """

    # Class attribute: the value used for unseen categories (oop_rules §7).
    UNKNOWN_FREQUENCY: float = 0.0

    def __init__(self, columns=None):
        self.columns = columns

    # ---- read-only accessors (oop_rules §6) -------------------------------
    @property
    def learned_vocabulary_size(self) -> dict:
        """How many distinct categories were seen per column, in training."""
        if not hasattr(self, "frequencies_"):
            raise AttributeError("FrequencyEncoder has not been fitted yet.")
        return {c: len(v) for c, v in self.frequencies_.items()}

    def fit(self, X: pd.DataFrame, y=None):
        if not isinstance(X, pd.DataFrame):
            raise TypeError(f"X must be a DataFrame, got {type(X).__name__}.")
        if self.columns is None:
            raise ValueError("FrequencyEncoder requires an explicit `columns` list.")

        missing = [c for c in self.columns if c not in X.columns]
        if missing:
            raise KeyError(f"Columns not found in X: {missing}")

        self.frequencies_ = {}
        for col in self.columns:
            self.frequencies_[col] = (
                X[col].value_counts(normalize=True).to_dict()
            )
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        if not hasattr(self, "frequencies_"):
            raise AttributeError("FrequencyEncoder must be fitted before transform().")
        if not isinstance(X, pd.DataFrame):
            raise TypeError(f"X must be a DataFrame, got {type(X).__name__}.")

        encoded = {}
        for col in self.columns:
            mapped = X[col].map(self.frequencies_[col])
            n_unseen = int(mapped.isna().sum())
            if n_unseen:
                self._unseen_counts = getattr(self, "_unseen_counts", {})
                self._unseen_counts[col] = n_unseen
            encoded[f"{col}_Freq"] = mapped.fillna(self.UNKNOWN_FREQUENCY).to_numpy()

        return pd.DataFrame(encoded, index=X.index)

    def get_feature_names_out(self, input_features=None):
        return np.array([f"{c}_Freq" for c in self.columns])

> **Caption:** A4: FrequencyEncoder source


## Screenshot — A5: ColumnTransformer/Pipeline source

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Column groups, derived from the actual dtypes.
numeric_cols = [c for c in X.columns if X[c].dtype != object]
categorical_cols = [c for c in X.columns if X[c].dtype == object]

# Rocket is handled by frequency encoding (Part D), so it is excluded from the
# one-hot group.
FREQ_COLS = ["Rocket"]
ONEHOT_COLS = [c for c in categorical_cols if c not in FREQ_COLS]

print(f"Numeric  ({len(numeric_cols)}): {numeric_cols}")
print(f"One-hot  ({len(ONEHOT_COLS)}): {ONEHOT_COLS}")
print(f"Frequency({len(FREQ_COLS)}): {FREQ_COLS}")

# --- Numeric branch: median impute (idle, see H.5) then standardize ---
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

# --- Categorical branch: impute, then one-hot with rare collapsing ---
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="infrequent_if_exist",  # unseen test categories are safe
        min_frequency=10,                      # collapse rare levels
        sparse_output=False,
    )),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_cols),
        ("cat", categorical_pipeline, ONEHOT_COLS),
        ("freq", FrequencyEncoder(columns=FREQ_COLS), FREQ_COLS),
    ],
    remainder="drop",
)

# The full pipeline: frequency-encode, then the column transformer.
# Constructed but NOT yet fitted.
full_pipeline = Pipeline([
    ("preprocessor", preprocessor),
])

print("\nPipeline constructed but NOT fitted.")
print("Fitting happens next, on X_train only.")

> **Caption:** A5: ColumnTransformer/Pipeline source


## Screenshot — A6: FrequencyEncoder vocabulary evidence

In [ ]:
# =====================================================================
# H.10c - CHECK 3: the frequency encoder used training frequencies only
# =====================================================================

freq_encoder = (
    full_pipeline.named_steps["preprocessor"]
    .named_transformers_["freq"]
)

print("FrequencyEncoder vocabulary, learned from training data:\n")
print(f"  Distinct rockets learned : {freq_encoder.learned_vocabulary_size['Rocket']}")
print(f"  Distinct rockets in full : {X['Rocket'].nunique()}")
print(f"  Distinct rockets in test : {X_test['Rocket'].nunique()}")

# Compare against frequencies computed from the FULL dataset - they must differ,
# proving the encoder did not see the test rows.
train_freq = X_train["Rocket"].value_counts(normalize=True)
full_freq = X["Rocket"].value_counts(normalize=True)

sample_rockets = train_freq.head(5).index.tolist()
freq_comparison = pd.DataFrame({
    "learned_from_train": [freq_encoder.frequencies_["Rocket"].get(r, 0.0) for r in sample_rockets],
    "actual_train_freq": [train_freq[r] for r in sample_rockets],
    "full_data_freq": [full_freq[r] for r in sample_rockets],
}, index=sample_rockets)

display(freq_comparison.round(6))

learned_matches_train = np.allclose(
    freq_comparison["learned_from_train"], freq_comparison["actual_train_freq"]
)
print(f"Learned values match TRAIN frequencies : {learned_matches_train}")
assert learned_matches_train, "FrequencyEncoder did not use training frequencies!"
print("PASS: frequency mapping came from the training split only.")

> **Caption:** A6: FrequencyEncoder vocabulary evidence


## Screenshot — A7: Feature origins breakdown

In [ ]:
# =====================================================================
# H.16 - CHART 3: where the features came from
# =====================================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Left: contribution of each source column to final width ---
# Count one-hot names per source column using the authoritative name list.
n_country = sum(1 for n in onehot_names if n.startswith("Country_"))
n_company = sum(1 for n in onehot_names if n.startswith("Company_"))

contributions = {
    f"Numeric ({len(numeric_cols)} cols)": len(numeric_cols),
    "Country one-hot": n_country,
    "Company one-hot": n_company,
    "Rocket frequency": 1,
}
total = sum(contributions.values())

bars = axes[0].barh(list(contributions.keys()), list(contributions.values()),
                    color=["#4C72B0", "#55A868", "#8172B2", "#DD8452"],
                    edgecolor="white", linewidth=1.2)
for bar, v in zip(bars, contributions.values()):
    axes[0].text(v + 0.4, bar.get_y() + bar.get_height()/2,
                 f"{v}  ({v/total*100:.0f}%)", va="center", fontsize=10, fontweight="bold")
axes[0].set_xlabel("Columns produced", fontsize=11)
axes[0].set_title(f"Final feature space: {total} columns", fontsize=12, fontweight="bold")
axes[0].set_xlim(0, max(contributions.values()) * 1.4)

# --- Right: the Rocket encoding decision, side by side ---
rocket_options = {
    "One-hot\n(min_frequency=10)": 86,
    "Frequency\nencoding (chosen)": 1,
}
bars2 = axes[1].bar(list(rocket_options.keys()), list(rocket_options.values()),
                    color=["#C44E52", "#55A868"], edgecolor="white", linewidth=1.5)
for bar, v in zip(bars2, rocket_options.values()):
    axes[1].text(bar.get_x() + bar.get_width()/2, v + 2, str(v),
                 ha="center", fontsize=14, fontweight="bold")
axes[1].set_ylabel("Columns added for 'Rocket'", fontsize=11)
axes[1].set_title("Part D's frequency-encoding decision, quantified",
                  fontsize=12, fontweight="bold")
axes[1].set_ylim(0, 100)
axes[1].annotate("86 columns for one feature\nwould dominate the matrix",
                 xy=(0, 86), xytext=(0.35, 62), fontsize=9.5, color="#7A2020",
                 arrowprops=dict(arrowstyle="->", color="#C44E52", lw=1.5))

plt.tight_layout()
plt.show()

print(f"Final matrix: {X_train_t.shape[1]} features from {len(X.columns)} source columns.")
print(f"Rows:features ratio = {len(X) // X_train_t.shape[1]} : 1")
print("\nFrequency encoding saved 85 columns versus one-hot on 'Rocket' alone.")
print("Part D committed to this; Part H implements it and shows the saving.")

> **Caption:** A7: Feature origins breakdown
